# 05 - Evaluation: Baseline vs Improved + Hallucination Detection

**Aligns with**: S4 Sec. 6 | **Estimated time**: 60 minutes | **Estimated cost**: ~$1.00 (full 30 questions x 2 pipelines)

## The question 04 left

We have a working RAG. **How do we prove it's better than quickstart? Better
than the previous version?**

## What this notebook does (around a real production accident)

In an earlier 5-question evaluation, the baseline pipeline answered
"Wells Fargo Q4 2025 net income" as **\$482M (the actual number is \$5.4B,
off by a factor of 10)**.

Three things in this notebook:
1. **Run all 30 questions**: Ragas 4 metrics on baseline (RecursiveChunker)
   to quantify failure modes
2. **Fix Q0**: switch to ParentChildChunker (child for retrieval, parent for
   the LLM), rerun, and see whether net income changes from wrong to right
3. **Add Hallucination Detector**: claim-level verification on the wrong Q0
   answer to demonstrate that **the bad \$482M can be detected automatically**

## The 30-question Golden Set

`data/golden_set/golden.jsonl`:

| Category | Count | What it tests |
|---|---|---|
| fact_finding | 10 | retrieve + extract a number |
| semantic | 8 | semantic alignment |
| single_doc_multihop | 4 | cross-page within one PDF |
| cross_doc | 4 | combine Tesla + AMD |
| out_of_corpus | 4 | refuse behavior |


In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
from dotenv import load_dotenv; load_dotenv(override=True)

from src.pipelines.ingestion import IngestionPipeline
from src.pipelines.query import QueryPipeline
from src.chunkers import RecursiveChunker, ParentChildChunker
from src.retrievers import VectorRetriever, BM25Retriever, HybridRetriever
from src.generators import RAGGenerator
from src.evaluators import RagasEvaluator, HallucinationDetector
from src.observability import CostTracker
import pandas as pd, collections, random

In [2]:
# Kernel check: this notebook must use the project .venv (not system Python).
# In Cursor/VS Code: select kernel → "Python (fin-rag-lab)" or
#   .venv/bin/python
# Then Restart Kernel and re-run from the top.
import pydantic, sys
from pydantic._internal import _typing_extra
print("python :", sys.executable)
print("pydantic:", pydantic.__version__)
assert "fin-rag-lab" in sys.executable or ".venv" in sys.executable, (
    f"Wrong kernel — expected project .venv, got:\n  {sys.executable}\n"
    "Select kernel: Python (fin-rag-lab)  then Restart Kernel."
)
assert hasattr(_typing_extra, "merge_cls_and_parent_ns"), (
    "Broken pydantic install (missing merge_cls_and_parent_ns). "
    "In the project root run:\n"
    "  .venv/bin/pip install --force-reinstall 'pydantic==2.9.2'"
)
print("env ok")

python : /Users/daniel/Desktop/USC/Job/直通硅谷/fin-rag-lab/.venv/bin/python
pydantic: 2.9.2
env ok


## Step 1: Load the golden set

In [6]:
golden = RagasEvaluator.load_golden_set(ROOT / "data/golden_set/golden.jsonl")
print(f"{len(golden)} questions")
for cat, n in collections.Counter(g["category"] for g in golden).most_common():
    print(f"  {cat:25s} {n}")
random.seed(0)
print("\nSample:")
for g in random.sample(golden, 3):
    print(f"  [{g['category']}] {g['question']}")

# Cost knob: SUBSET vs FULL.
#   SUBSET (5 questions, ~$0.10/pipeline)  -> fast iteration
#   FULL  (30 questions, ~$0.50/pipeline) -> real evaluation
USE_FULL = True   # flip to False for the 5-question fast loop
EVAL_SET = golden if USE_FULL else golden[:5]
print(f"\nWill evaluate on {len(EVAL_SET)} questions (USE_FULL={USE_FULL})")

30 questions
  fact_finding              10
  semantic                  8
  single_doc_multihop       4
  cross_doc                 4
  out_of_corpus             4

Sample:
  [out_of_corpus] What is the current US federal funds rate?
  [semantic] How does Tesla describe progress on its AI and robotics initiatives?
  [cross_doc] Are Tesla and AMD both reporting growth in their AI-related businesses?

Will evaluate on 30 questions (USE_FULL=True)


## Step 2: Build the BASELINE pipeline - RecursiveChunker

The "naive default": 400-token chunks, no parent-child. This is the version
that failed Q0 in our earlier 5-question test.

In [8]:
cost_baseline = CostTracker()
pipeline = IngestionPipeline(cost_tracker=cost_baseline)
docs = [pipeline.ingest(str(ROOT / f"data/uploads/{n}.pdf")).document
        for n in ["wells_fargo", "tesla", "amd"]]

# Baseline: recursive chunker, no parent-child
recursive_chunks = []
for d in docs:
    recursive_chunks.extend(RecursiveChunker(chunk_size=400, overlap=60).chunk(d))

vec_b = VectorRetriever(
    persist_dir=str(ROOT / "tmp_chroma_05_baseline"), collection="lab_05_baseline",
    embeddings_cache_dir=ROOT / "cache/embeddings",
)
vec_b.reset(); vec_b.index(recursive_chunks)
bm_b = BM25Retriever(); bm_b.index(recursive_chunks)
hybrid_b = HybridRetriever(vec_b, bm_b)   # no parent_store
qp_baseline = QueryPipeline(hybrid_b, RAGGenerator(cost_tracker=cost_baseline), cost_tracker=cost_baseline)
print(f"baseline pipeline ready ({len(recursive_chunks)} chunks)")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


baseline pipeline ready (1317 chunks)


## Step 3: Run Ragas on baseline

This makes ~150-180 LLM calls (4 metrics x ~30 questions x 1-2 calls each).
At judge-tier model pricing, expect ~$0.40-0.50.

In [9]:
def baseline_query(question: str):
    result = qp_baseline.query(question)
    return {"answer": result["answer"], "chunks": result["chunks"]}

evaluator = RagasEvaluator()
print(f"Evaluating BASELINE on {len(EVAL_SET)} questions...\n")
baseline_df = evaluator.evaluate(baseline_query, EVAL_SET, verbose=True)

# Save raw results
baseline_df.to_csv(ROOT / "tmp_baseline_ragas.csv", index=False)
print(f"\nSaved to tmp_baseline_ragas.csv")
print(f"Baseline cost so far: ${cost_baseline.total:.4f}")
baseline_df.head(8)

Evaluating BASELINE on 30 questions...

  [1/30] What was Wells Fargo's Q4 2025 net income?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  [2/30] What was Wells Fargo's diluted EPS in Q4 2025?
  [3/30] What was Wells Fargo's CET1 ratio at the end of Q4 2025?
  [4/30] What was Tesla's vehicle production in Q1 2026?
  [5/30] What was Tesla's energy storage deployment in Q1 2026?
  [6/30] What was AMD's data center segment revenue in Q4 2025?
  [7/30] What was AMD's gaming segment revenue in Q4 2025?
  [8/30] What was AMD's full-year 2025 revenue?
  [9/30] What was AMD's gross margin in Q4 2025?
  [10/30] What was Wells Fargo's average loan balance in Q4 2025?
  [11/30] How did Wells Fargo's Consumer Banking and Lending segment p...
  [12/30] What were the main drivers of Wells Fargo's net interest inc...
  [13/30] How does Tesla describe progress on its AI and robotics init...
  [14/30] What were Tesla's key operational highlights for Q1 2026?
  [15/30] How did AMD's Instinct GPU products perform in 2025?
  [16/30] What is AMD's outlook for the data center AI market?
  [17/30] What capital return actions did Wells Fargo t

Evaluating: 100%|██████████| 120/120 [00:46<00:00,  2.61it/s]


Saved to tmp_baseline_ragas.csv
Baseline cost so far: $0.1100


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall,category
0,What was Wells Fargo's Q4 2025 net income?,[[Section: Q4 2024 Q4 2025 > FY 2025 Revenue]\...,What I found in the sources: Source 2 lists nu...,Wells Fargo reported Q4 2025 net income of $5....,0.800000,0.000000,0.500000,1.0,fact_finding
1,What was Wells Fargo's diluted EPS in Q4 2025?,"[News Release | January 14, 2026 Wells Fargo R...",Wells Fargo's diluted EPS in Q4 2025 was $1.62...,Diluted EPS of $1.62 per share in Q4 2025.,1.000000,1.000000,1.000000,1.0,fact_finding
2,What was Wells Fargo's CET1 ratio at the end o...,"[[Section: Q4 2024 Q4 2025]\n$1.53, [Section: ...",What I found in the sources: The materials sta...,Wells Fargo reported its Common Equity Tier 1 ...,0.428571,0.000000,0.250000,1.0,fact_finding
3,What was Tesla's vehicle production in Q1 2026?,[[Section: A I & SOFTWARE > Cumulative Paid Ro...,"What I found in the sources: Source 1 states ""...",Tesla's Q1 2026 update reports the quarter's p...,0.583333,0.000000,0.000000,0.0,fact_finding
4,What was Tesla's energy storage deployment in ...,[[Section: SUMMARY HIGHLIGHTS > Q1-2025 Q2-202...,Tesla's energy storage deployment in Q1 2026 w...,Tesla's energy storage deployments are reporte...,1.000000,1.000000,1.000000,1.0,fact_finding
5,What was AMD's data center segment revenue in ...,[[Section: Q4 2024 Q4 2025 > Q4 2025 Data Cent...,AMD's data center segment revenue in Q4 2025 w...,AMD's Q4 2025 earnings reports data center seg...,1.000000,0.962497,0.583333,0.0,fact_finding
6,What was AMD's gaming segment revenue in Q4 2025?,[[Section: Q4 2024 Q4 2025 > Client & Gaming]\...,"What I found in the sources: Source 1 shows ""C...",AMD's Q4 2025 release reports gaming segment r...,0.428571,0.000000,1.000000,1.0,fact_finding
7,What was AMD's full-year 2025 revenue?,[[Section: AMD Financial Results]\nFourth Quar...,"AMD's full-year 2025 revenue was $34,639M [^3].",AMD's Q4 2025 release includes full-year 2025 ...,1.000000,1.000000,0.000000,1.0,fact_finding


## Step 4: Score baseline by category - find the weak spots

In [10]:
metric_cols = [c for c in baseline_df.columns if c in
               ("faithfulness","answer_relevancy","context_precision","context_recall")]

if metric_cols and "category" in baseline_df.columns:
    by_cat = baseline_df.groupby("category")[metric_cols].mean().round(3)
    print("Baseline metrics by category:\n")
    print(by_cat)
    print("\nThe lowest-scoring category is your highest-leverage improvement target.")
    
    # Find specific failures (faithfulness < 0.5 means likely-wrong answer)
    failures = baseline_df[baseline_df["faithfulness"] < 0.5]
    if len(failures):
        print(f"\n{len(failures)} likely-wrong-answer cases (faithfulness < 0.5):")
        for _, row in failures.head(3).iterrows():
            print(f"\n   Q:   {row['user_input'][:80]}")
            print(f"   A:   {str(row['response'])[:120]}")
            print(f"   Ref: {str(row.get('reference', ''))[:120]}")

Baseline metrics by category:

                     faithfulness  answer_relevancy  context_precision  \
category                                                                 
cross_doc                   0.866             0.218              0.062   
fact_finding                0.791             0.488              0.533   
out_of_corpus               0.499             0.000              0.156   
semantic                    0.848             0.494              0.469   
single_doc_multihop         0.854             0.669              0.390   

                     context_recall  
category                             
cross_doc                     0.250  
fact_finding                  0.700  
out_of_corpus                 0.250  
semantic                      0.375  
single_doc_multihop           0.750  

The lowest-scoring category is your highest-leverage improvement target.

5 likely-wrong-answer cases (faithfulness < 0.5):

   Q:   What was Wells Fargo's CET1 ratio at the end of Q4

## Step 5: Build IMPROVED pipeline - ParentChildChunker + parent expansion

Hypothesis: Q0 failed because the recursive chunker fragmented Wells Fargo's
financial table. Each 400-token chunk had A net-income number, but the wrong
one (a single segment, or Q4 2024). The LLM picked one and ran with it.

The fix: small **child** chunks for retrieval (precision), large **parent**
chunks for the LLM (context including column headers and the section title).

`HybridRetriever` already supports this: pass `parent_store={parent_id: parent_chunk}`
and it auto-swaps children for parents at retrieval time.

In [11]:
cost_improved = CostTracker()

pc = ParentChildChunker(parent_size=800, child_size=150)
all_parents, all_children = [], []
for d in docs:
    parents, children = pc.chunk_with_parents(d)
    all_parents.extend(parents)
    all_children.extend(children)
parent_store = {p.chunk_id: p for p in all_parents}

print(f"  parents: {len(all_parents)}  children: {len(all_children)}")

# Index CHILDREN for retrieval (small, precise), use parent_store at retrieval time
vec_i = VectorRetriever(
    persist_dir=str(ROOT / "tmp_chroma_05_improved"), collection="lab_05_improved",
    embeddings_cache_dir=ROOT / "cache/embeddings",
)
vec_i.reset(); vec_i.index(all_children)
bm_i = BM25Retriever(); bm_i.index(all_children)
hybrid_i = HybridRetriever(vec_i, bm_i, parent_store=parent_store)  # parent expansion ON
qp_improved = QueryPipeline(hybrid_i, RAGGenerator(cost_tracker=cost_improved), cost_tracker=cost_improved)
print(f"improved pipeline ready (retrieval over {len(all_children)} children, generation gets {len(all_parents)} parents)")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


  parents: 85  children: 578
improved pipeline ready (retrieval over 578 children, generation gets 85 parents)


## Step 6: Run Ragas on improved pipeline (same questions)

In [12]:
def improved_query(question: str):
    result = qp_improved.query(question)
    return {"answer": result["answer"], "chunks": result["chunks"]}

print(f"Evaluating IMPROVED on {len(EVAL_SET)} questions...\n")
improved_df = evaluator.evaluate(improved_query, EVAL_SET, verbose=True)
improved_df.to_csv(ROOT / "tmp_improved_ragas.csv", index=False)
print(f"\nSaved to tmp_improved_ragas.csv")
print(f"Improved cost so far: ${cost_improved.total:.4f}")

Evaluating IMPROVED on 30 questions...

  [1/30] What was Wells Fargo's Q4 2025 net income?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  [2/30] What was Wells Fargo's diluted EPS in Q4 2025?
  [3/30] What was Wells Fargo's CET1 ratio at the end of Q4 2025?
  [4/30] What was Tesla's vehicle production in Q1 2026?
  [5/30] What was Tesla's energy storage deployment in Q1 2026?
  [6/30] What was AMD's data center segment revenue in Q4 2025?
  [7/30] What was AMD's gaming segment revenue in Q4 2025?
  [8/30] What was AMD's full-year 2025 revenue?
  [9/30] What was AMD's gross margin in Q4 2025?
  [10/30] What was Wells Fargo's average loan balance in Q4 2025?
  [11/30] How did Wells Fargo's Consumer Banking and Lending segment p...
  [12/30] What were the main drivers of Wells Fargo's net interest inc...
  [13/30] How does Tesla describe progress on its AI and robotics init...
  [14/30] What were Tesla's key operational highlights for Q1 2026?
  [15/30] How did AMD's Instinct GPU products perform in 2025?
  [16/30] What is AMD's outlook for the data center AI market?
  [17/30] What capital return actions did Wells Fargo t

Evaluating: 100%|██████████| 120/120 [00:47<00:00,  2.50it/s]


Saved to tmp_improved_ragas.csv
Improved cost so far: $0.1618


## Step 7: Side-by-side comparison

In [13]:
# Aggregate by category, compare
b_by = baseline_df.groupby("category")[metric_cols].mean().round(3)
i_by = improved_df.groupby("category")[metric_cols].mean().round(3)
delta = (i_by - b_by).round(3)

print("BASELINE  (RecursiveChunker, no parent expansion):")
print(b_by)
print("\nIMPROVED  (ParentChildChunker + parent expansion):")
print(i_by)
print("\nDELTA  (improved - baseline)  positive = better:")
print(delta)

# Headline: did Q0 get fixed?
q0_b = baseline_df[baseline_df["user_input"].str.contains("Q4 2025 net income", case=False, na=False)]
q0_i = improved_df[improved_df["user_input"].str.contains("Q4 2025 net income", case=False, na=False)]
if len(q0_b) and len(q0_i):
    print("\n" + "="*70)
    print("Q0 - the production accident from earlier")
    print("="*70)
    print(f"Q: {q0_b.iloc[0]['user_input']}")
    print(f"   Reference: {str(q0_b.iloc[0].get('reference',''))[:100]}")
    print(f"\n   BASELINE  answer: {str(q0_b.iloc[0]['response'])[:120]}")
    print(f"             faithfulness={q0_b.iloc[0]['faithfulness']:.2f}  context_recall={q0_b.iloc[0]['context_recall']:.2f}")
    print(f"\n   IMPROVED  answer: {str(q0_i.iloc[0]['response'])[:120]}")
    print(f"             faithfulness={q0_i.iloc[0]['faithfulness']:.2f}  context_recall={q0_i.iloc[0]['context_recall']:.2f}")

BASELINE  (RecursiveChunker, no parent expansion):
                     faithfulness  answer_relevancy  context_precision  \
category                                                                 
cross_doc                   0.866             0.218              0.062   
fact_finding                0.791             0.488              0.533   
out_of_corpus               0.499             0.000              0.156   
semantic                    0.848             0.494              0.469   
single_doc_multihop         0.854             0.669              0.390   

                     context_recall  
category                             
cross_doc                     0.250  
fact_finding                  0.700  
out_of_corpus                 0.250  
semantic                      0.375  
single_doc_multihop           0.750  

IMPROVED  (ParentChildChunker + parent expansion):
                     faithfulness  answer_relevancy  context_precision  \
category                              

**How to read the results**:
- `delta` is the improvement matrix. `fact_finding` should improve most -
  parent expansion lets the LLM see column headers and the section title,
  preventing pull-the-wrong-number errors
- If a category gets worse, parent_size=800 may be pulling in too much noise.
  In production you tune by query distribution
- Q0 should move from \$482M toward \$5.4B. If it's still \$482M, the
  retriever still didn't find the correct table


## Step 8: Hallucination Detector - claim-level validation on Q0

Ragas faithfulness gives one aggregate score. For compliance, you need to know
**which specific claim was wrong**. `HallucinationDetector` decomposes the
answer into atomic claims and verifies each against retrieved context.

This is the safety net for cases where Ragas faithfulness is 0.5
(half-hallucinated): you need to know which half.

In [14]:
detector = HallucinationDetector(cost_tracker=cost_improved)
sample_q = "What was Wells Fargo's Q4 2025 net income?"

# Run on baseline (the one that hallucinated $482M) to demo the detector catching it
baseline_result = qp_baseline.query(sample_q)
print(f"Q: {sample_q}")
print(f"BASELINE answer: {baseline_result['answer']}\n")

if baseline_result["chunks"]:
    report = detector.detect(baseline_result["answer"], baseline_result["chunks"])
    print(f"Decomposed into {report.n_claims} claims  |  Faithfulness: {report.faithfulness_score:.0%}")
    print(f"   entailed: {report.n_entailed}  unsupported: {report.n_unsupported}  refuted: {report.n_refuted}\n")
    for cv in report.claims:
        print(f"   [{cv.verdict.upper():12s}] {cv.claim}")
        print(f"      reasoning: {cv.reasoning[:120]}")

Q: What was Wells Fargo's Q4 2025 net income?
BASELINE answer: What I found in the sources: Source 2 lists net income figures in a row: "Wells Fargo net income $ 5,361 5,589 5,079 (4) 6 $ 21,338 19,722" [^2]. Source 3 shows "$1.09" (appears separately, likely EPS) [^3].

What's missing: The context does not explicitly label which of those numbers (e.g., $5,361, $5,589, $5,079) corresponds to Q4 2025, so I cannot confidently state the Q4 2025 net income. I would need the table headers or an explicit line identifying "Q4 2025 net income: $X" to answer.

Decomposed into 8 claims  |  Faithfulness: 88%
   entailed: 7  unsupported: 1  refuted: 0

   [ENTAILED    ] Source 2 lists net income figures in a row.
      reasoning: The context shows a section labeled “Noninterest income” with a row containing multiple net income figures listed across
   [ENTAILED    ] The row in Source 2 includes the numbers 5,361, 5,589, 5,079, 21,338, and 19,722.
      reasoning: The context’s Noninterest income r

## Step 9: OOC refusal validation (compliance)

In [ ]:
ooc_qs = [g for g in golden if g["category"] == "out_of_corpus"]
print("Out-of-corpus refusal test (improved pipeline):\n" + "-"*70)
hits, misses = 0, 0
for g in ooc_qs:
    result = qp_improved.query(g["question"])
    verification = detector.detect(result["answer"], result.get("chunks", []))
    refused_correctly = result.get("outcome") == "refuse" and verification.n_refuted + verification.n_unsupported == 0
    mark = "REFUSED  " if refused_correctly else "ANSWERED!"
    if refused_correctly: hits += 1
    else: misses += 1
    print(f"\n[{mark}] {g['question']}")
    print(f"   answer: {result['answer'][:160]}")
print(f"\nOOC refusal rate: {hits}/{hits+misses} = {hits/max(1,hits+misses):.0%}")
print("Baseline:", cost_baseline.summary_line())
print("Improved + detector:", cost_improved.summary_line())

## Interview talking point (with real numbers from your run)

> "I built a 30-question golden set across 5 categories (fact_finding,
> semantic, multi-hop, cross-doc, out-of-corpus). On baseline (RecursiveChunker
> 400/60), Ragas faithfulness was X%. **One specific failure** - Wells Fargo
> Q4 2025 net income - was answered as \$482M instead of \$5.4B. Diagnosis:
> the recursive chunker fragmented the financial table, and the LLM picked the
> wrong segment number.
>
> I switched to ParentChildChunker (150-token children for retrieval,
> 800-token parents fed to the LLM). The same Q0 then answered correctly.
> Across all 30 questions, faithfulness improved from X% to Y%.
> **A net-income hallucination of \$482M -> \$5.4B is the kind of error that
> blocks production deployment in any regulated industry.** I added the
> claim-level HallucinationDetector as a compliance safety net specifically
> for this failure mode."

## Exercise 5.A

Swap the chunker to fixed_size and run the same evaluation. Compare the metric
matrices across all 3 chunkers. **Don't trust "recursive is the best default"
until your own data confirms it.**
